Install Libraries

In [ ]:
!pip install -q kagglehub

Import Libraries

In [ ]:
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import roc_curve
from sklearn.metrics import auc

from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input

from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.layers import BatchNormalization

from tensorflow.keras.models import Model

from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau

Download Dataset

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "paultimothymooney/chest-xray-pneumonia"
)

print(dataset_path)

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
/kaggle/input/chest-xray-pneumonia


Dataset Paths

In [ ]:
dataset_path = os.path.join(
    dataset_path,
    "chest_xray"
)

train_dir = os.path.join(dataset_path,"train")

test_dir = os.path.join(dataset_path,"test")

val_dir = os.path.join(dataset_path,"val")

Hyperparameters

In [ ]:
IMG_SIZE = (224,224)

BATCH_SIZE = 32

SEED = 42

Training Dataset

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(

    train_dir,

    validation_split=0.2,

    subset="training",

    seed=SEED,

    image_size=IMG_SIZE,

    batch_size=BATCH_SIZE
)

Found 5216 files belonging to 2 classes.
Using 4173 files for training.


Validation Dataset

In [ ]:
val_ds = tf.keras.utils.image_dataset_from_directory(

    train_dir,

    validation_split=0.2,

    subset="validation",

    seed=SEED,

    image_size=IMG_SIZE,

    batch_size=BATCH_SIZE
)

Found 5216 files belonging to 2 classes.
Using 1043 files for validation.


Test Dataset

In [ ]:
test_ds = tf.keras.utils.image_dataset_from_directory(

    test_dir,

    shuffle=False,

    image_size=IMG_SIZE,

    batch_size=BATCH_SIZE
)

Found 624 files belonging to 2 classes.


Optimize Dataset

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_densenet = train_ds.map(
    lambda x,y:(preprocess_input(x),y),
    num_parallel_calls=AUTOTUNE
)

val_densenet = val_ds.map(
    lambda x,y:(preprocess_input(x),y),
    num_parallel_calls=AUTOTUNE
)

test_densenet = test_ds.map(
    lambda x,y:(preprocess_input(x),y),
    num_parallel_calls=AUTOTUNE
)

train_densenet = train_densenet.prefetch(AUTOTUNE)

val_densenet = val_densenet.prefetch(AUTOTUNE)

test_densenet = test_densenet.prefetch(AUTOTUNE)

Compute Class Weights

In [ ]:
# Get labels from training dataset
y_train = np.concatenate([y.numpy() for x, y in train_ds], axis=0)

# Compute class weights
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = {
    i: weight for i, weight in enumerate(class_weights)
}

print("Class Weights:")
print(class_weight_dict)

Class Weights:
{0: np.float64(1.9107142857142858), 1: np.float64(0.6772151898734177)}


Load Pre-trained DenseNet121

In [ ]:
base_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

print("Base model loaded successfully!")

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Base model loaded successfully!


Build Transfer Learning Model

In [ ]:
inputs = tf.keras.Input(shape=(224, 224, 3))

x = base_model(inputs, training=False)

x = GlobalAveragePooling2D()(x)

x = BatchNormalization()(x)

x = Dropout(0.4)(x)

x = Dense(
    256,
    activation="relu"
)(x)

x = Dropout(0.3)(x)

outputs = Dense(
    1,
    activation="sigmoid"
)(x)

model = Model(
    inputs,
    outputs
)

Display Model Summary

In [ ]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,304,257 (27.86 MB)

 Trainable params: 264,705 (1.01 MB)

 Non-trainable params: 7,039,552 (26.85 MB)

Compile Model

In [ ]:
model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),

    loss="binary_crossentropy",

    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

Verify Model

In [ ]:
print("="*50)

print("Model Compiled Successfully")

print("="*50)

print("Trainable Layers:", len(model.trainable_variables))

print("="*50)

Model Compiled Successfully
Trainable Layers: 6


Callbacks

In [ ]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "best_densenet121.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=8,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

callbacks = [
    checkpoint,
    early_stop,
    reduce_lr
]

Train Only the Classification Head

In [ ]:
history = model.fit(

    train_densenet,

    validation_data=val_densenet,

    epochs=10,

    class_weight=class_weight_dict,

    callbacks=callbacks
)

Epoch 1/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 376ms/step - accuracy: 0.7291 - auc: 0.8375 - loss: 0.4837 - precision: 0.9060 - recall: 0.6975
Epoch 1: val_accuracy improved from None to 0.87536, saving model to best_densenet121.keras

Epoch 1: finished saving model to best_densenet121.keras
131/131 ━━━━━━━━━━━━━━━━━━━━ 130s 702ms/step - accuracy: 0.8222 - auc: 0.9379 - loss: 0.3324 - precision: 0.9635 - recall: 0.7890 - val_accuracy: 0.8754 - val_auc: 0.9845 - val_loss: 0.3181 - val_precision: 0.9985 - val_recall: 0.8375 - learning_rate: 1.0000e-04
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step - accuracy: 0.9181 - auc: 0.9795 - loss: 0.1930 - precision: 0.9867 - recall: 0.9002
Epoch 2: val_accuracy improved from 0.87536 to 0.94439, saving model to best_densenet121.keras

Epoch 2: finished saving model to best_densenet121.keras
131/131 ━━━━━━━━━━━━━━━━━━━━ 42s 322ms/step - accuracy: 0.9228 - auc: 0.9825 - loss: 0.1772 - precision: 0.9859 - recall: 0.9085 - val_accuracy: 0.9444 - va

Fine-Tune DenseNet121

In [ ]:
base_model.trainable = True

# Freeze all layers except the last 40
for layer in base_model.layers[:-40]:
    layer.trainable = False

print("Trainable Layers:",
      sum([layer.trainable for layer in base_model.layers]))

Trainable Layers: 40


Recompile for Fine-Tuning

In [ ]:
model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),

    loss="binary_crossentropy",

    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

Fine-Tune the Model

In [ ]:
fine_history = model.fit(

    train_densenet,

    validation_data=val_densenet,

    epochs=20,

    initial_epoch=history.epoch[-1] + 1,

    class_weight=class_weight_dict,

    callbacks=callbacks
)

Epoch 11/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step - accuracy: 0.9600 - auc: 0.9922 - loss: 0.1075 - precision: 0.9853 - recall: 0.9594
Epoch 11: val_accuracy did not improve from 0.96644
131/131 ━━━━━━━━━━━━━━━━━━━━ 127s 650ms/step - accuracy: 0.9621 - auc: 0.9933 - loss: 0.0971 - precision: 0.9886 - recall: 0.9598 - val_accuracy: 0.9569 - val_auc: 0.9963 - val_loss: 0.0952 - val_precision: 0.9947 - val_recall: 0.9484 - learning_rate: 1.0000e-05
Epoch 12/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 240ms/step - accuracy: 0.9620 - auc: 0.9947 - loss: 0.0868 - precision: 0.9907 - recall: 0.9572
Epoch 12: val_accuracy did not improve from 0.96644
131/131 ━━━━━━━━━━━━━━━━━━━━ 76s 319ms/step - accuracy: 0.9626 - auc: 0.9942 - loss: 0.0901 - precision: 0.9893 - recall: 0.9598 - val_accuracy: 0.9616 - val_auc: 0.9965 - val_loss: 0.0885 - val_precision: 0.9948 - val_recall: 0.9547 - learning_rate: 1.0000e-05
Epoch 13/20
130/131 ━━━━━━━━━━━━━━━━━━━━ 0s 248ms/step - accuracy: 0.9691 - auc: 0.9955 - l

Load the Best Model

In [ ]:
best_model = tf.keras.models.load_model(
    "best_densenet121.keras"
)

print("Best DenseNet121 model loaded successfully!")

Best DenseNet121 model loaded successfully!


Save Final Model

In [ ]:
best_model.save("DenseNet121_Final_Model.keras")

print("Model saved successfully!")

Model saved successfully!


In [ ]:
# Evaluate the best model on the test dataset
loss, accuracy, precision, recall, auc = best_model.evaluate(
    test_densenet,
    verbose=1
)

print("\n" + "="*50)
print("        DenseNet121 Test Performance")
print("="*50)
print(f"Test Accuracy : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Test Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"Test Recall   : {recall:.4f} ({recall*100:.2f}%)")
print(f"Test AUC      : {auc:.4f}")
print(f"Test Loss     : {loss:.4f}")
print("="*50)

20/20 ━━━━━━━━━━━━━━━━━━━━ 36s 809ms/step - accuracy: 0.8686 - auc: 0.9573 - loss: 0.3694 - precision: 0.8319 - recall: 0.9897

        DenseNet121 Test Performance
Test Accuracy : 0.8686 (86.86%)
Test Precision: 0.8319 (83.19%)
Test Recall   : 0.9897 (98.97%)
Test AUC      : 0.9573
Test Loss     : 0.3694
